In [1]:
"""
======================================================================
GDELT BIGQUERY EXPORT: CLEAN, ORGANIZE & SPLIT
======================================================================
Takes the raw BigQuery CSV export and produces clean, deduplicated
master URL lists split by language, ready for bulk text extraction.

INPUT:  The raw CSV you exported from BigQuery UI
OUTPUT: Two clean CSVs (English + Hindi) in your project directories
"""
import pandas as pd
import numpy as np
import os

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    pass

# ==========================================
# 1. CONFIGURATION
# ==========================================
# UPDATE THIS PATH to wherever you saved the BigQuery export in Drive
BQ_EXPORT_PATH = '/content/drive/MyDrive/bq-results-20260702-144222-1783003513513/bq-results-20260702-144222-1783003513513.csv'

# Output directories
ENG_DIR = '/content/drive/MyDrive/BMP/BMP - English'
HIN_DIR = '/content/drive/MyDrive/BMP/BMP - Hindi'
os.makedirs(ENG_DIR, exist_ok=True)
os.makedirs(HIN_DIR, exist_ok=True)

# ==========================================
# 2. LOAD RAW EXPORT
# ==========================================
print("Loading raw BigQuery export...")
df_raw = pd.read_csv(BQ_EXPORT_PATH)
print(f"Raw rows loaded: {len(df_raw):,}")
print(f"Columns detected: {df_raw.columns.tolist()}")

# Normalize column names (BigQuery exports can vary)
col_map = {}
for col in df_raw.columns:
    cl = col.lower().strip()
    if 'date' in cl and 'identifier' not in cl:
        col_map[col] = 'DATE'
    elif 'documentidentifier' in cl or (cl == 'url'):
        col_map[col] = 'url'
    elif 'title' in cl:
        col_map[col] = 'title'
    elif 'source' in cl or 'domain' in cl:
        col_map[col] = 'domain'

df_raw = df_raw.rename(columns=col_map)

# Ensure we have the minimum required columns
assert 'url' in df_raw.columns, f"Cannot find URL column. Available: {df_raw.columns.tolist()}"
if 'DATE' not in df_raw.columns:
    # Try to find any column that looks like a GDELT integer date
    for col in df_raw.columns:
        sample = str(df_raw[col].iloc[0])
        if len(sample) >= 8 and sample[:8].isdigit():
            df_raw['DATE'] = df_raw[col]
            break

print(f"✓ Columns mapped. Working with: {df_raw.columns.tolist()}")

# ==========================================
# 3. PARSE DATES & DEDUPLICATE
# ==========================================
# GDELT DATE is integer: YYYYMMDDHHMMSS (14 digits)
# We only need the first 8 digits (YYYYMMDD) for daily resolution

df_raw['date_str'] = df_raw['DATE'].astype(str).str.strip()

# Handle both 14-digit (YYYYMMDDHHMMSS) and 8-digit (YYYYMMDD) formats
def parse_gdelt_date(s):
    s = str(s).strip()
    if len(s) >= 8 and s[:8].isdigit():
        return pd.to_datetime(s[:8], format='%Y%m%d', errors='coerce')
    return pd.NaT

df_raw['date'] = df_raw['date_str'].apply(parse_gdelt_date)

# Drop rows with missing dates or URLs
df_raw = df_raw.dropna(subset=['date', 'url'])

# Deduplicate by URL (keep first occurrence)
before_dedup = len(df_raw)
df_raw = df_raw.drop_duplicates(subset=['url'], keep='first')
print(f"✓ Deduplication: {before_dedup:,} → {len(df_raw):,} unique URLs "
      f"({before_dedup - len(df_raw):,} duplicates removed)")

# ==========================================
# 4. NEWSPAPER & LANGUAGE MAPPING
# ==========================================
def map_newspaper(url):
    url = str(url).lower()
    # English
    if 'thehindu.com' in url:
        return 'Hindu', 'english'
    if 'timesofindia.indiatimes.com' in url:
        return 'TOI', 'english'
    if 'indianexpress.com' in url:
        return 'Express', 'english'
    if 'hindustantimes.com' in url:
        return 'Hindustan Times', 'english'
    if 'newindianexpress.com' in url:
        return 'New Indian Express', 'english'
    # Hindi
    if 'livehindustan.com' in url:
        return 'Hindustan', 'hindi'
    if 'navbharattimes.indiatimes.com' in url:
        return 'NBT', 'hindi'
    if 'amarujala.com' in url:
        return 'AmarUjala', 'hindi'
    if 'jagran.com' in url:
        return 'Dainik Jagran', 'hindi'
    if 'bhaskar.com' in url or 'dainikbhaskar.com' in url:
        return 'Dainik Bhaskar', 'hindi'
    return None, None

mapped = df_raw['url'].apply(map_newspaper)
df_raw['newspaper'] = mapped.apply(lambda x: x[0])
df_raw['language'] = mapped.apply(lambda x: x[1])

# Drop unmapped URLs (syndication links, CDN assets, etc.)
df_mapped = df_raw.dropna(subset=['newspaper']).copy()
unmapped_count = len(df_raw) - len(df_mapped)
print(f"✓ Newspaper mapping: {len(df_mapped):,} mapped, {unmapped_count:,} unmapped URLs dropped")

# ==========================================
# 5. STUDY WINDOW FILTER
# ==========================================
study_start = pd.to_datetime('2023-07-01')
study_end = pd.to_datetime('2024-08-01')

df_study = df_mapped[(df_mapped['date'] >= study_start) & (df_mapped['date'] <= study_end)].copy()
print(f"✓ Study window filter (Jul 2023 – Jul 2024): {len(df_study):,} articles retained")

# ==========================================
# 6. ORGANIZE & SAVE
# ==========================================
output_cols = ['date', 'newspaper', 'language', 'url']
if 'title' in df_study.columns:
    output_cols.append('title')

# Split by language
df_eng = df_study[df_study['language'] == 'english'].copy()
df_hin = df_study[df_study['language'] == 'hindi'].copy()

# Save English
eng_out = f'{ENG_DIR}/GDELT_EXHAUSTIVE_ENGLISH_URLS.csv'
df_eng[output_cols].to_csv(eng_out, index=False)

# Save Hindi
hin_out = f'{HIN_DIR}/GDELT_EXHAUSTIVE_HINDI_URLS.csv'
df_hin[output_cols].to_csv(hin_out, index=False)

# Save Combined Master
master_out = f'{ENG_DIR}/GDELT_EXHAUSTIVE_ALL_URLS.csv'
df_study[output_cols].to_csv(master_out, index=False)

# ==========================================
# 7. FINAL SUMMARY
# ==========================================
print("\n" + "=" * 60)
print("CLEANUP COMPLETE — ORGANIZED CORPUS SUMMARY")
print("=" * 60)
print(f"Date Range: {df_study['date'].min().date()} to {df_study['date'].max().date()}")
print(f"\nTotal Unique Articles: {len(df_study):,}")
print(f"\n--- English Panel ({len(df_eng):,} articles) ---")
print(df_eng['newspaper'].value_counts().to_string())
print(f"\n--- Hindi Panel ({len(df_hin):,} articles) ---")
print(df_hin['newspaper'].value_counts().to_string())

print(f"\n--- Output Files ---")
print(f"English: {eng_out}")
print(f"Hindi:   {hin_out}")
print(f"Master:  {master_out}")

# ==========================================
# 8. WEEKLY VOLUME PREVIEW (Sanity Check)
# ==========================================
print("\n--- WEEKLY VOLUME PREVIEW (First 4 Weeks, English) ---")
df_eng['week'] = df_eng['date'].dt.isocalendar().week.astype(int)
df_eng['year'] = df_eng['date'].dt.year
weekly_preview = df_eng.groupby(['year', 'week', 'newspaper']).size().unstack(fill_value=0)
print(weekly_preview.head(4).to_string())

print("\n✓ Ready for Phase 2: Trafilatura Bulk Text Extraction")

Mounted at /content/drive
Loading raw BigQuery export...
Raw rows loaded: 1,078,388
Columns detected: ['DATE', 'url']
✓ Columns mapped. Working with: ['DATE', 'url']
✓ Deduplication: 1,078,388 → 1,078,388 unique URLs (0 duplicates removed)
✓ Newspaper mapping: 1,078,388 mapped, 0 unmapped URLs dropped
✓ Study window filter (Jul 2023 – Jul 2024): 1,078,388 articles retained

CLEANUP COMPLETE — ORGANIZED CORPUS SUMMARY
Date Range: 2023-07-01 to 2024-08-01

Total Unique Articles: 1,078,388

--- English Panel (626,292 articles) ---
newspaper
TOI                225322
Hindu              152827
Express            135170
Hindustan Times    112973

--- Hindi Panel (452,096 articles) ---
newspaper
AmarUjala         144484
NBT               115744
Hindustan          93747
Dainik Jagran      49063
Dainik Bhaskar     49058

--- Output Files ---
English: /content/drive/MyDrive/BMP/BMP - English/GDELT_EXHAUSTIVE_ENGLISH_URLS.csv
Hindi:   /content/drive/MyDrive/BMP/BMP - Hindi/GDELT_EXHAUSTIVE_HINDI_